# 56 — Prompt Engineering
**Goal:** Design effective prompts for resume-domain LLM tasks.

Ch. 55 gave us a client; this chapter gives it something worth saying. An LLM's output quality is bounded by the **prompt contract**: the role, the examples, the context, and the output format you specify. For resume work the contract has to fight the model's default behavior — vague verbs, unquantified claims, free-form prose — so every prompt in this chapter is built from the same five components: system role, few-shot examples, context, instruction, and output format.

**Why it matters for resumes / ATS:** prompts are the only place the ATS's domain knowledge (STAR format, quantified results, skill categories) meets the model. A classifier told to "return only the category" is parseable; one left to free-associate produces junk that needs cleaning. Prompt structure is the cheapest quality lever in the whole block — no retraining, no new models, just disciplined templates.

![LLM Prompt Engineering Pipeline](../../../assets/images/llm_prompt_pipeline_1785491212825.png)

> **Figure:** The LLM prompt pipeline — from resume+JD context through versioned templates, OpenRouter API, to validated structured JSON output.

The figure is the road map for the rest of this block: resume + JD text enters a **template** (this chapter), the template is versioned so changes are trackable (Ch. 57), the call goes out through the single OpenRouter client from Ch. 55, and the response is validated against a JSON contract (Ch. 58) before any downstream consumer touches it. Each stage is a chapter; the pipeline as a whole is what turns raw text into dependable structured resume data.

## 1. Prompt Structure Patterns

Every resume-task prompt decomposes into the same five parts, in the same order: **system** (role + constraints), **few-shot examples** (demonstrate the pattern), **context** (the resume data), **instruction** (what to do with it), and **output format** (the shape of the answer). Omitting any one invites a specific failure: no system role — the model argues with you; no examples — format drift; no output format — prose you must parse.

**What the code does:** prints the five components and the four canonical resume task patterns — `Classification` ("Classify this skill as technical/soft/tool"), `Extraction` ("Extract years of experience"), `Rewriting` ("Rewrite this bullet using STAR"), `Generation` ("Generate interview questions"). These four task types map 1:1 onto Ch. 55's tier policy — classification is the cheap task, generation the flagship one.

In [ ]:
print('''Prompt components for resume tasks:
1. System prompt — role, constraints, output format
2. Few-shot examples — demonstrate the pattern
3. Context — the resume data being processed
4. Instruction — what to do with the context
5. Output format — JSON schema or format specification

Example patterns:
- Classification: "Classify this skill as technical/soft/tool"
- Extraction: "Extract years of experience from this text"
- Rewriting: "Rewrite this bullet point using STAR format"
- Generation: "Generate interview questions based on this resume"''')

## 2. Building a Prompt Template System

Templates are **parameterized prompts**: a fixed skeleton with slots for the input. `PROMPT_TEMPLATES` stores each template's `system` role and `examples` (input/output pairs), and `format_prompt()` assembles a full prompt from them — so the same template serves any input text with zero prompt-writing per call.

**What the code does:**
- `skill_classify` — the system role demands one of four categories and "Return only the category"; four examples pin the taxonomy (`Python — technical`, `Team Leadership — soft`, `TensorFlow — tool`, `NLP — domain_knowledge`).
- `bullet_rewrite` — the system role is a "professional resume coach" rewriting to quantified STAR; one example shows the transformation ("Responsible for ML models" — "Developed ML models achieving 95% accuracy...").
- `format_prompt(template, input)` — builds `System: ...`, then one `Input:/Output:` block per example, then the live `Input: {text}\nOutput:` ending.

**Expected (verified by running):** formatting `bullet_rewrite` with "Was responsible for data pipelines" yields a prompt that ends in `Input: Was responsible for data pipelines\nOutput:` — the model is expected to continue after the colon, which is exactly how you get a single answer instead of a chatty one.

In [ ]:
PROMPT_TEMPLATES = {
    "skill_classify": {
        "system": "You are a resume skill classifier. Classify each skill as: technical, soft, tool, or domain_knowledge. Return only the category.",
        "examples": [
            ("Python", "technical"),
            ("Team Leadership", "soft"),
            ("TensorFlow", "tool"),
            ("NLP", "domain_knowledge"),
        ]
    },
    "bullet_rewrite": {
        "system": "You are a professional resume coach. Rewrite resume bullets using STAR format (Situation, Task, Action, Result). Make them quantified and impactful.",
        "examples": [
            ("Responsible for ML models", "Developed ML models achieving 95% accuracy, reducing manual review time by 40%"),
        ]
    },
}

def format_prompt(template_name, input_text):
    """Build a complete prompt from template."""
    tpl = PROMPT_TEMPLATES.get(template_name)
    if not tpl: return input_text
    
    parts = [f"System: {tpl['system']}"]
    for ex_in, ex_out in tpl.get("examples", []):
        parts.append(f"Input: {ex_in}\nOutput: {ex_out}")
    parts.append(f"Input: {input_text}\nOutput:")
    return "\n\n---\n\n".join(parts)

print(format_prompt("bullet_rewrite", "Was responsible for data pipelines"))

## 3. Testing Prompts

Prompts are code — they regress. The discipline here is borrowed from software testing: a **golden test set** of 20—50 known (input — expected output) pairs, run against every prompt variant, scored on accuracy, consistency, robustness, and cost.

**What the code does:** prints the methodology: build the golden set, run each variant against it, score outputs (exact match, semantic similarity, or human rating), and track versions in a registry. The four test dimensions matter for different reasons: `Accuracy` is extraction correctness, `Consistency` is same-input-same-output (LLMs are stochastic, so you measure variance), `Robustness` is behavior on empty or malformed input (a resume parser will receive garbage), and `Cost` is tokens per call — measurable with Ch. 55's price table.

**Try it:** run a golden set through `format_prompt()` from section 2 and eyeball two outputs per input — identical prompts should produce near-identical answers; if not, your temperature or prompt is too loose.

In [ ]:
print('''Prompt testing methodology:
1. Golden test set — 20-50 examples with known correct outputs
2. Run each prompt variant against the test set
3. Score outputs: exact match, semantic similarity, human rating
4. Track prompt versions in a registry

Testing dimensions:
- Accuracy: Does it produce correct extractions?
- Consistency: Same input = same output?
- Robustness: What happens with edge cases (empty, malformed)?
- Cost: Token usage per call''')

## Summary: Structured prompt templates improve consistency. Test prompts systematically against golden datasets.

**A prompt is a function — parameterize it, and test it like one.**

Five components (system, examples, context, instruction, format) become a reusable template, and every template ships with a golden set that measures accuracy, consistency, robustness, and cost. That combination — templating plus tests — is what makes LLM behavior *predictable enough* to ship: the model still samples, but the variance is bounded by the contract you wrote.

Templates that get tested tend to get edited, which is exactly the problem Ch. 57 solves: versioning prompts in a registry with regression checks.